In [ ]:
from langgraph.graph import StateGraph, START,END
from typing import TypedDict,Annotated
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from langchain_community.chat_models.oci_generative_ai import ChatOCIGenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
import operator
from langchain_openai import ChatOpenAI


In [ ]:
load_dotenv()

model = ChatOpenAI()

In [44]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

In [ ]:
def chat_node(state: ChatState):
    messages = state.get("messages")

    response = model.invoke(messages)

    return {"messages" : [response]}

In [40]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

graph.add_node("chat_node",chat_node)

graph.add_edge(START,"chat_node")
graph.add_edge("chat_node",END)

chatbot = graph.compile(checkpointer=checkpointer)

In [ ]:
initial_state = {"messages": [HumanMessage(content="what is the capital of india?")]}

chatbot.invoke(initial_state)['messages'][-1].content

In [41]:
thread_id = 1
while True:
    user_message = input("Type here: ")
    print(f"User: {user_message}")

    if user_message.strip().lower() in ["quite","bye","exit"]:
        break

    config = {"configurable":{"thread_id":thread_id}}
    response = chatbot.invoke({"messages" : [HumanMessage(content=user_message)]},config=config)

    print(f"AI: {response["messages"][-1].content}")

User: Hi, My name is Harsh
AI: Hello Harsh! It's nice to meet you. Is there something I can help you with or would you like to chat?
User: What is my name?
AI: Your name is Harsh! You told me that just a minute ago.
User: add 10 with 100
AI: The result of adding 10 to 100 is:

100 + 10 = 110
User: add 25
AI: Adding 25 to 110:

110 + 25 = 135
User: bye


In [42]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='Hi, My name is Harsh', additional_kwargs={}, response_metadata={}, id='e35af626-76cc-43f5-8545-0b249263a3c9'), AIMessage(content="Hello Harsh! It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={'finish_reason': 'stop', 'time_created': '2026-03-25 12:11:23.166000+00:00'}, response_metadata={'model_id': 'meta.llama-4-scout-17b-16e-instruct', 'model_version': '1.0.0', 'request_id': '06E90E83F9364D688D2A918FAD259A1C/C91E3FFAE017B0E2FAB1CE777E84730E/6356C78E57B529539410D4BDC111E81D', 'content-length': '345', 'finish_reason': 'stop', 'time_created': '2026-03-25 12:11:23.166000+00:00'}, id='lc_run--019d24e8-1174-7713-899d-86e759695c7c-0', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='8379ceb1-d8b7-4a3e-8b24-9a9b47e9fb4d'), AIMessage(content='Your name is Harsh! You told me that just a minute ago.',